# Notebook 11.1  An Arabic text-to-speech front end and an objective intelligibility check

**Goal.** Take Arabic text through normalization, diacritization, and grapheme-to-phoneme (G2P), show how alternative diacritizations change the phoneme sequence, then synthesize with a pretrained model and measure intelligibility with an automatic recognizer.

**What runs here.** The front end (number normalization, a diacritization demo, and a rule-based G2P) and the Word Error Rate evaluation are real and run with no downloads. Synthesis itself needs a pretrained model, so that cell is a clearly marked hook with a synthetic fallback. This accompanies Chapter 11; the point is the *front end*, which decides Arabic TTS quality.

## 1. Setup

In [ ]:
import re, unicodedata
print('ready')

## 2. Text normalization: numbers to spoken words

The front end must expand digits, dates, and symbols into words before pronunciation. Here is a small integer-to-Arabic-words normalizer for 0-999 (real systems handle gender agreement, dates, and currency).

In [ ]:
ONES = ['صفر','واحد','اثنان','ثلاثة','أربعة','خمسة','ستة','سبعة','ثمانية','تسعة','عشرة',
        'أحد عشر','اثنا عشر','ثلاثة عشر','أربعة عشر','خمسة عشر','ستة عشر','سبعة عشر','ثمانية عشر','تسعة عشر']
TENS = {20:'عشرون',30:'ثلاثون',40:'أربعون',50:'خمسون',60:'ستون',70:'سبعون',80:'ثمانون',90:'تسعون'}
HUND = {100:'مئة',200:'مئتان',300:'ثلاثمئة',400:'أربعمئة',500:'خمسمئة',600:'ستمئة',700:'سبعمئة',800:'ثمانمئة',900:'تسعمئة'}

def number_to_arabic(n):
    n = int(n)
    if n < 20: return ONES[n]
    if n < 100:
        t, o = (n//10)*10, n%10
        return TENS[t] if o==0 else ONES[o] + ' و' + TENS[t]
    h, rem = (n//100)*100, n%100
    return HUND[h] if rem==0 else HUND[h] + ' و' + number_to_arabic(rem)

def normalize_numbers(text):
    return re.sub(r'\d+', lambda m: number_to_arabic(m.group()), text)

print(normalize_numbers('عندي 3 كتب و25 قلمًا'))
for k in [0,7,15,42,100,256,999]: print(k, '->', number_to_arabic(k))

## 3. The diacritization problem

Arabic is written without short vowels, so one spelling maps to several spoken words. Restoring the diacritics (tashkīl) is the hardest front-end step; real systems use trained models, but the ambiguity is easy to show by hand. The bare form علم has at least three readings.

In [ ]:
readings = {
    'عِلْم':  ('ilm',    'knowledge'),
    'عَلَم':  ('alam',   'flag'),
    'عَلَّمَ': ('allama', 'he taught'),
}
for dia,(translit,gloss) in readings.items():
    print(f'{dia}   /{translit}/   {gloss}')

## 4. Grapheme-to-phoneme from diacritized text

Once the diacritics are present, Arabic G2P is largely rule-governed. This simplified converter maps consonants and the short-vowel diacritics to phonemes, doubles a consonant under shadda (gemination), and treats alif as a long /aː/. It is a teaching version, not production-grade.

In [ ]:
CONS = {'ب':'b','ت':'t','ث':'θ','ج':'dʒ','ح':'ħ','خ':'x','د':'d','ذ':'ð','ر':'r','ز':'z',
        'س':'s','ش':'ʃ','ص':'sˤ','ض':'dˤ','ط':'tˤ','ظ':'ðˤ','ع':'ʕ','غ':'ɣ','ف':'f','ق':'q',
        'ك':'k','ل':'l','م':'m','ن':'n','ه':'h','و':'w','ي':'j','ء':'ʔ','أ':'ʔ','إ':'ʔ','ة':'t'}
FATHA,KASRA,DAMMA,SUKUN,SHADDA = '\u064E','\u0650','\u064F','\u0652','\u0651'
SHORT = {FATHA:'a', KASRA:'i', DAMMA:'u'}
MARKS = {FATHA,KASRA,DAMMA,SUKUN,SHADDA}

def g2p(text):
    out, i = [], 0
    while i < len(text):
        ch = text[i]
        if ch == 'ا':
            out.append('aː'); i += 1; continue
        if ch in CONS:
            ph = CONS[ch]; i += 1
            # collect all combining marks attached to this consonant, in any order
            marks = []
            while i < len(text) and text[i] in MARKS:
                marks.append(text[i]); i += 1
            if SHADDA in marks: ph = ph + 'ː'      # gemination
            out.append(ph)
            for mk in marks:
                if mk in SHORT: out.append(SHORT[mk])
            continue
        i += 1  # skip anything else
    return out

print('عِلْم  ->', g2p('عِلْم'))
print('عَلَم  ->', g2p('عَلَم'))
print('عَلَّمَ ->', g2p('عَلَّمَ'))

### Why the front end matters

The three diacritizations of the *same* letters produce three different phoneme sequences, so a wrong diacritization makes a perfect acoustic model fluently say the wrong word.

In [ ]:
for dia in readings:
    print(f'{dia:8s} -> /{" ".join(g2p(dia))}/')

## 5. Synthesis hook and the Word Error Rate cross-check

The objective intelligibility check runs synthesized speech through an automatic recognizer and measures the Word Error Rate (WER) against the input text. The WER function below is real; `synthesize` is a hook for a pretrained model, with a fallback so the evaluation logic runs without audio.

**To synthesize for real:** load a pretrained Arabic TTS (for example a FastSpeech 2 + HiFi-GAN or an XTTS checkpoint), generate a waveform from the phonemes, transcribe it with an Arabic recognizer (Whisper), and pass the transcript to `wer` below.

In [ ]:
def wer(reference, hypothesis):
    r, h = reference.split(), hypothesis.split()
    # Levenshtein distance over words
    d = [[0]*(len(h)+1) for _ in range(len(r)+1)]
    for i in range(len(r)+1): d[i][0] = i
    for j in range(len(h)+1): d[0][j] = j
    for i in range(1,len(r)+1):
        for j in range(1,len(h)+1):
            cost = 0 if r[i-1]==h[j-1] else 1
            d[i][j] = min(d[i-1][j]+1, d[i][j-1]+1, d[i-1][j-1]+cost)
    return d[len(r)][len(h)] / max(1,len(r))

def synthesize(text):
    raise NotImplementedError('Plug in a pretrained Arabic TTS here to produce a waveform.')

# Fallback demo of the evaluation loop (no audio): a 'recognized' transcript with one slip
intended   = 'أريد رصيد حسابي اليوم'
recognized = 'أريد رصيد حسابي اليون'   # stand-in for an ASR transcript of the synthesized speech
print('intended  :', intended)
print('recognized:', recognized)
print('WER = %.1f%%' % (100*wer(intended, recognized)))

## 6. Where to go next

- Replace the hand-written diacritization demo with a trained Arabic diacritizer, and the toy G2P with a full rule set (sun/moon assimilation, hamza, tanwīn).
- Plug a real Arabic TTS into `synthesize`, transcribe with an Arabic recognizer, and report WER on a held-out set, per dialect.
- Run a small Mean Opinion Score listening test for naturalness alongside the objective WER check (Section 11.8).
- For recitation, add tajwīd-aware rules and expert evaluation (Section 11.7).